<a href="https://colab.research.google.com/github/SujahathMSM/PdfToLinkedInPosts/blob/dev/AutomatedLinkedIPosts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2 openai==0.28.0 ipywidgets

In [ ]:
#Uploading a file in colab
from google.colab import files
uploaded = files.upload()

Saving Improving Your Memory (DK Essential Managers) by DK, David Thomas (z-lib.org).pdf to Improving Your Memory (DK Essential Managers) by DK, David Thomas (z-lib.org) (1).pdf


In [ ]:
#import all the required libraries
import PyPDF2
import openai  # LLM and image generation (DALL-E)
from ipywidgets import Dropdown, Button, VBox, Output  #interactive UI
import matplotlib.pyplot as plt  # Displaying the generated image
import requests
from PIL import Image
from io import BytesIO

#Define the API key
openai.api_key = "YOUR_API_KEY"


In [ ]:
# Text extraction from the pdf files

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        num_pages = len(reader.pages)
        print(f"PDF has {num_pages} pages. Extracting text...")
        for i, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
            if (i+1) % 100 == 0:
                print(f"Processed {i+1} pages...")
    return text

# Split text into chunks (according to character count)
def chunk_text(text, chunk_size=5000):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [ ]:
# Summarize a chunk of text using an LLM (GPT)
def summarize_text(text_chunk):
    prompt = f"Summarize the following text concisely:\n\n{text_chunk}"
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4-turbo",  # (also can use 3.5-turbo)
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            max_tokens=150
        )
        summary = response['choices'][0]['message']['content']
        return summary.strip()
    except Exception as e:
        print("Error during summarization:", e)
        return ""

In [ ]:
# Aggregate summaries from all text chunks
def aggregate_summaries(pdf_text, chunk_size=5000):
    print("Splitting PDF text into chunks...")
    chunks = chunk_text(pdf_text, chunk_size)
    summaries = []
    print(f"Total chunks to summarize: {len(chunks)}")
    for idx, chunk in enumerate(chunks):
        print(f"Summarizing chunk {idx+1}/{len(chunks)}...")
        summary = summarize_text(chunk)
        summaries.append(summary)
    aggregated_summary = "\n".join(summaries)
    return aggregated_summary

In [ ]:
# Generate post ideas
def generate_post_ideas(aggregated_summary):
    prompt = (
        "Based on the following summarized content, generate 10 unique, creative, and professional ideas "
        "for a LinkedIn post. Each idea should be concise:\n\n"
        f"{aggregated_summary}\n\n"
        "List the ideas, numbered 1 to 10:"
    )
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4-turbo",  # Using GPT-4 Turbo
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=500
        )
        ideas = response['choices'][0]['message']['content']
        return ideas.strip()
    except Exception as e:
        print("Error generating post ideas:", e)
        return ""

In [ ]:
# Generate a full LinkedIn post from a selected idea and aggregated summary using GPT-4 Turbo
def generate_linkedin_post(selected_idea, aggregated_summary):
    prompt = (
        f"Using the idea: '{selected_idea}', and considering the following context:\n\n"
        f"{aggregated_summary}\n\n"
        "Write a detailed, engaging, and professional LinkedIn post."
    )
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4-turbo",  # Using GPT-4 Turbo
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=500
        )
        linkedin_post = response['choices'][0]['message']['content']
        return linkedin_post.strip()
    except Exception as e:
        print("Error generating LinkedIn post:", e)
        return ""

In [ ]:
#Generate AI image
def generate_ai_image(linkedin_post):
    # detailed image prompt
    image_prompt = (
        "Generate a high-resolution, modern digital illustration that conveys a professional and innovative atmosphere. "
        "Incorporate a clean, minimalistic design with vibrant colors and dynamic composition. "
        "The illustration should evoke creativity and success, suitable for a professional setting. "
        "Do not include any text or lettering in the image."
    )


    api_url = "https://api.openai.com/v1/images/generations"


    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {openai.api_key}"
    }

    # Prepare the JSON payload
    payload = {
        "prompt": image_prompt,
        "n": 1,
        "size": "1024x1024"
    }

    try:
        response = requests.post(api_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        image_url = data['data'][0]['url']
        return image_url
    except Exception as e:
        print("Error during image generation:", e)
        return None

In [ ]:
#Main Execution part of the program

#Enter the uploaded file name here
pdf_path = "Improving Your Memory (DK Essential Managers) by DK, David Thomas (z-lib.org).pdf"


pdf_text = extract_text_from_pdf(pdf_path)


print("Aggregating summaries from PDF text...")
aggregated_summary = aggregate_summaries(pdf_text, chunk_size=5000)
print("Aggregated Summary Generated.\n")


print("Generating LinkedIn post ideas...")
ideas_text = generate_post_ideas(aggregated_summary)
print("Post Ideas Generated:\n", ideas_text)


ideas_list = []
for line in ideas_text.split("\n"):
    line = line.strip()
    if line and (line[0].isdigit() or line.startswith("-")):
        # Remove numbering or bullet symbols if present
        idea = line.lstrip("0123456789.- ").strip()
        ideas_list.append(idea)

dropdown = Dropdown(options=ideas_list, description='Select Idea:')
button = Button(description="Generate LinkedIn Post")
output = Output()

def on_button_click(b):
    with output:
        output.clear_output()
        selected_idea = dropdown.value
        print("Selected Idea:", selected_idea, "\n")

        # Inform the user that the LinkedIn post is being generated.
        print("Generating LinkedIn post... please wait.")
        linkedin_post = generate_linkedin_post(selected_idea, aggregated_summary)
        print("\nGenerated LinkedIn Post:")
        print(linkedin_post)

        # Inform the user that the AI image is being generated.
        print("\nGenerating AI image... please wait.")
        image_url = generate_ai_image(linkedin_post)
        if image_url:
            print("\nAI-generated Image URL:", image_url)
            # Display the image if accessible
            try:
                img_data = requests.get(image_url).content
                img = Image.open(BytesIO(img_data))
                plt.figure(figsize=(8, 6))
                plt.imshow(img)
                plt.axis('off')
                plt.show()
            except Exception as e:
                print("Error displaying image:", e)
        else:
            print("\nNo image was generated.")

button.on_click(on_button_click)

# Display the interactive widget box
display(VBox([dropdown, button, output]))

PDF has 76 pages. Extracting text...
Aggregating summaries from PDF text...
Splitting PDF text into chunks...
Total chunks to summarize: 25
Summarizing chunk 1/25...
Error during summarization: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Summarizing chunk 2/25...
Error during summarization: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Summarizing chunk 3/25...
Error during summarization: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
Summarizing chunk 4/25...
Error during summarization: You exceeded your current quota, please check your plan and billing 